In [ ]:
# =========================================
# VaR HISTÓRICO (HS) — Simulación Histórica
# =========================================
# Método no paramétrico: estima el VaR como el cuantil empírico (percentil 99)
# de las pérdidas históricas observadas en la ventana de entrenamiento.
# No asume ninguna distribución paramétrica para los rendimientos.
#
# Configuración (igual que el modelo MLP para comparación justa):
#   - Ventana de entrenamiento: 900 observaciones históricas
#   - Reentrenamiento: diario (el cuantil se recalcula cada sesión)
# =========================================
import numpy as np
import pandas as pd

# Cargamos el dataset preprocesado (generado en datos.ipynb)
dataset = pd.read_pickle("../data/dataset_tfg.pkl").copy().sort_index()

alpha = 0.99       # nivel de confianza del VaR (99%)
retrain_every = 1  # recalculamos el cuantil cada día

eval_start = pd.Timestamp("2015-01-01")
eval_end   = pd.Timestamp("2024-12-31")
eval_dates = dataset.loc[eval_start:eval_end].index

var_preds = []
real_targets = []
dates = []

current_var = None  # almacena el último VaR válido calculado

for i, current_date in enumerate(eval_dates):

    if i % retrain_every == 0:
        # Seleccionamos las 900 observaciones anteriores al día actual
        train_end = current_date - pd.Timedelta(days=1)
        train_df  = dataset.loc[:train_end].tail(900)

        loss_train = train_df["target"].dropna().values

        if len(loss_train) >= 250:
            # VaR estimado como el percentil 99 de las pérdidas históricas observadas
            # Cuantil empírico puro: sin ajuste paramétrico ni interpolación
            current_var = np.quantile(loss_train, alpha)

    if current_var is None:
        continue  # saltamos si aún no tenemos suficiente historia

    # Registramos la predicción del VaR y la pérdida real del día
    test_loss = dataset.loc[current_date, "target"]
    var_preds.append(current_var)
    real_targets.append(test_loss)
    dates.append(current_date)

# ==============================
# Resultados y métricas básicas
# ==============================
out_hs = pd.DataFrame(
    {"VaR_HS": np.array(var_preds), "loss_real": np.array(real_targets)},
    index=pd.to_datetime(dates)
).sort_index()

eval_df_hs = out_hs.loc["2015":"2024"].dropna().copy()

# Indicador de violación: 1 si la pérdida real supera el VaR estimado (exceedance)
eval_df_hs["viol"] = (eval_df_hs["loss_real"] > eval_df_hs["VaR_HS"]).astype(int)
eval_df_hs["year"] = eval_df_hs.index.year

# Tasa de violación global: debe aproximarse al 1% para una buena calibración
viol_rate = eval_df_hs["viol"].mean()

# Desglose anual: permite detectar años con clustering de violaciones (p.ej. COVID-2020)
yearly = eval_df_hs.groupby("year")["viol"].mean()

print("\n=== VaR Histórico (HS) ===")
print(f"Ventana: 900 obs | retrain cada {retrain_every} día(s)")
print(f"Total preds      : {len(eval_df_hs)}")
print(f"Exceedance rate  : {viol_rate:.4f}")
print(f"Esperado         : {1-alpha:.4f}")

print("\nViolaciones por año (HS):")
for y, v in yearly.items():
    print(f"{y}: {v:.4f}")

# Exportamos para el análisis comparativo en comp_final.ipynb
eval_df_hs.to_csv("../data/hs_var_predictions.csv")
print("\nGuardado: hs_var_predictions.csv")